# 02 — Feature Engineering

Implements **plan §6** and §7. Demonstrates the `FeatureEngineer` transformer
from `src.features`, validates it against the train/val/test splits from
`src.data`, and saves processed parquet files for the modeling notebook to
reload without re-running EDA-grade computations.

Plan §6.5 calls out: *"All feature engineering is implemented as a sklearn
Pipeline so the same transformations applied at training are applied at
inference. This is the single most important defence against train-serve
skew in a small project."* This notebook is the smoke test for that contract.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import (
    DEFAULT_DATA_PATH, FEATURE_COLS, TARGET_COL, imbalance_ratio,
    load_raw, save_splits, split_stratified, split_time_based,
)
from src.features import FeatureConfig, FeatureEngineer

sns.set_theme(context='notebook', style='whitegrid', palette='muted')

## 1. Load and split (plan §7)

In [ ]:
df = load_raw(ROOT / DEFAULT_DATA_PATH)
splits_strat = split_stratified(df, random_state=42)
print('Stratified split:')
print(splits_strat.summary().to_string(index=False))
print(f'Imbalance ratio on train: {imbalance_ratio(splits_strat.y_train):.2f}')

splits_time = split_time_based(df, random_state=42)
print('\nTime-based split (production-mimicking):')
print(splits_time.summary().to_string(index=False))
print('Note: late slice has very few fraud cases — the §7.1 tradeoff to discuss.')

## 2. Fit the FeatureEngineer on TRAIN only (§6.4 critical)

In [ ]:
fe = FeatureEngineer(FeatureConfig(
    add_hour_of_day=True,
    add_is_night=True,
    add_log_amount=True,
    add_amount_zero_flag=True,
    add_amount_bucket=True,
    n_amount_buckets=10,
))
fe.fit(splits_strat.X_train)

print('Output columns:', len(fe.feature_names_out_))
print(fe.feature_names_out_)
print('\nAmount bucket edges (fitted on TRAIN only):')
print(fe.amount_bucket_edges_)

In [ ]:
# Transform all three splits with the SAME fitted transformer.
X_train_fe = fe.transform(splits_strat.X_train)
X_val_fe = fe.transform(splits_strat.X_val)
X_test_fe = fe.transform(splits_strat.X_test)

print(X_train_fe.shape, X_val_fe.shape, X_test_fe.shape)
X_train_fe.head()

## 3. Sanity-check the engineered features

In [ ]:
# log_amount distribution by class.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
y = splits_strat.y_train.values
axes[0].hist(X_train_fe.loc[y == 0, 'log_amount'], bins=80,
             alpha=0.6, label='genuine', color='steelblue', density=True)
axes[0].hist(X_train_fe.loc[y == 1, 'log_amount'], bins=80,
             alpha=0.7, label='fraud', color='crimson', density=True)
axes[0].set_title('log_amount by class')
axes[0].legend()

# Bucket distribution by class.
bucket_by_class = (
    pd.crosstab(X_train_fe['amount_bucket'], splits_strat.y_train, normalize='columns')
    .rename(columns={0: 'genuine', 1: 'fraud'})
)
bucket_by_class.plot(kind='bar', ax=axes[1])
axes[1].set_title('amount_bucket distribution by class')
axes[1].set_xlabel('bucket')
plt.tight_layout()
plt.show()
bucket_by_class

In [ ]:
# Hour-of-day fraud rate on the engineered training split.
by_hour = pd.DataFrame({
    'hour': X_train_fe['hour_of_day'].values,
    'fraud': splits_strat.y_train.values,
}).groupby('hour').agg(n=('fraud', 'size'), frauds=('fraud', 'sum'))
by_hour['fraud_rate'] = by_hour['frauds'] / by_hour['n']
by_hour.head(24)

## 4. Train-serve skew check

The same fitted `FeatureEngineer` must produce identical output for the same
input row whether called from `src.train` or from `src.api`. Quick check:
transforming a single row by itself must equal the corresponding row from a
batch transform.

In [ ]:
row = splits_strat.X_test.iloc[[0]]
out_single = fe.transform(row)
out_batch = fe.transform(splits_strat.X_test).iloc[[0]]

diff = (out_single.values - out_batch.values).max()
print(f'Max single-vs-batch diff: {diff}')
assert diff == 0, 'FeatureEngineer is not stateless across single vs batch calls!'
print('OK — same transformation regardless of batch size.')

## 5. Persist splits for the modeling notebook

In [ ]:
out_dir = ROOT / 'data' / 'processed' / 'stratified'
paths = save_splits(splits_strat, out_dir)
for name, p in paths.items():
    print(f'{name}: {p}')

out_dir_t = ROOT / 'data' / 'processed' / 'time'
paths_t = save_splits(splits_time, out_dir_t)
for name, p in paths_t.items():
    print(f'{name}: {p}')

## Notes

- This notebook does **not** fit any scaler. Tree-based models don't need it
  (§6.4) and the modeling notebook (`03_modeling`) is XGBoost-led. If you add
  a Logistic Regression baseline, attach a `ColumnTransformer` with
  `RobustScaler` on `seconds_since_start`, `amount`, `log_amount` and
  `StandardScaler` on V1..V28 — fit on TRAIN only.
- Interaction features are off by default (§6.3). To enable, pass
  `interaction_pairs=(("V14","V12"), ("V14","V10"))` into `FeatureConfig`.